# SDA-DSC-212 — Time Series Analysis and Forecasting
## Lab 3 — Classical Baselines: SARIMAX and ETS

**Duration** 50 minutes · **Work in pairs** · `git checkout lab3-start`

> The one rule for this whole course: every feature, every split and every scaler
> must be computable using only data that existed at the forecast origin.

### Tasks

| min | task |
|---|---|
| 5 | Aggregate the hourly series to **daily max** — the reserve-margin question is about the peak. Restrict to 2022 onward, build the exogenous frame (`cdd`, `hdd`, `is_weekend`, `is_ramadan`, `is_eid`), hold out the last 28 days. |
| 12 | Fit SARIMAX `(2,1,1)(1,1,1)[7]`. Explain why `m=7` on a daily series rather than `m=24` on the hourly one. Run `residual_diagnostics` and report Ljung-Box honestly. |
| 10 | Fit ETS with `fit_ets` (damped trend, `seasonal_periods=7`). Sweep the taxonomy with `ets_grid` by AIC. State when AIC comparisons are valid. |
| 10 | Forecast 28 days from both with 90% intervals. SARIMAX needs the **future** exogenous rows. Plot both forecasts against the held-out actuals. |
| 8 | MAE / MAPE / MASE and 90% coverage for each. Fit a no-exogenous SARIMA to price what temperature is worth. Compute the seasonal-naive baseline. |
| 5 | Commit. |

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

DATA = "../data"

Lab 1's output, reloaded — the labs are independent notebooks.

In [ ]:
from tayyar.data.load import load_demand, load_calendar, join_calendar

df = load_demand(f"{DATA}/ksa_grid_demand.csv")
df = join_calendar(df, load_calendar(f"{DATA}/ksa_calendar.csv"))
y = df["demand_mw"].astype(float)

print(f"hourly rows: {len(y):,}   from {y.index[0]} to {y.index[-1]}")

### Task 1 — the daily-max series, the exogenous frame, and the hold-out

Two decisions here, both of which encode the question rather than the data.

**Daily *max*, not daily mean.** The deliverable is a reserve margin. A control room sizes
reserve against the peak the system must carry, not against the average it happens to run at.
Aggregating with `.mean()` would answer a question nobody asked, and would answer it about
5,000 MW too low.

**`m = 7` on a daily series, not `m = 24` on the hourly one.** This is a wall you should walk
into deliberately. SARIMA's state space grows with the seasonal period: at `m=24` the
state vector is large enough that estimation slows from seconds to many minutes, and
`auto_arima` with `m=24` may simply never converge on this series. The classical route
therefore models the *daily* series with a weekly cycle, `m=7`, and leaves the sub-daily
structure to the feature-based ML route in Module 4. That hand-off is not an admission of
defeat; it is the honest boundary of the classical toolkit, and knowing where it sits is
part of the learning outcome.

In [ ]:
# TODO:
#   1. aggregate y to a DAILY series. Which aggregation answers a reserve-margin question?
#   2. restrict to 2022 onward.
#   3. build the exogenous frame with exactly these columns:
#        cdd        = max(daily max temp - 21, 0)     cooling-degree signal
#        hdd        = max(14 - daily min temp, 0)     heating-degree signal
#        is_weekend, is_ramadan, is_eid               daily flags, as int
#   4. hold out the last 28 days as `te`; everything before is `tr`. Slice the exog to match.
#
# HINT: aggregation encodes the question. `.mean()` and `.max()` are not two ways of saying
#       the same thing — the reserve margin is sized against the peak. Note also that cdd
#       uses the daily MAX temperature and hdd the daily MIN: each degree-day should be
#       built from the extreme that actually drives the load.

### Task 2 — SARIMAX (2,1,1)(1,1,1)[7], and an honest residual check

In [ ]:
from tayyar.models.sarimax import fit_sarimax, forecast_sarimax, residual_diagnostics

# TODO:
#   1. fit SARIMAX on `tr` with `Xtr`, order=(2,1,1), seasonal_order=(1,1,1,7).
#   2. print the exogenous coefficients — each is MW per unit of that regressor, so they
#      are readable out loud to an engineer. Sanity-check the sign of every one.
#   3. run residual_diagnostics(sar, lags=14) and print it in full.
#   4. report the Ljung-Box result HONESTLY, whatever it says. Do not quietly move on.
#
# HINT: do not be tempted to raise m to 24 and fit the hourly series. The state space grows
#       with m; the fit goes from seconds to many minutes and may not converge at all. The
#       daily series with m=7 is the classical route; sub-daily structure is Module 4's job.

### Task 3 — ETS, and the rules for reading an AIC

In [ ]:
from tayyar.models.ets import fit_ets, forecast_ets, ets_grid

# TODO:
#   1. fit_ets(tr, seasonal_periods=7). The defaults are trend="add", damped=True,
#      seasonal="add" — an ETS(A,Ad,A). Print alpha, beta, gamma, phi and the AIC.
#   2. run ets_grid(tr, seasonal_periods=7) and read the AIC ranking.
#   3. write one sentence on when those AIC values may be compared and when they may not.
#
# HINT: AIC is comparable ONLY across models fitted to the same target — same
#       transformation, same length, same differencing. Comparing the AIC of a model fitted
#       to log(y) against one fitted to y, or a d=0 model against a d=1 model, is comparing
#       likelihoods computed on different data. The number will still print. It will still
#       be meaningless.

### Task 4 — 28-day forecasts with 90% intervals

`get_forecast(steps=28, exog=...)` needs **exactly 28 rows of future exogenous values**. That
requirement is not an API inconvenience, it is the definition of an exogenous regressor:
temperature qualifies only because a day-ahead weather forecast exists. If you have to wait
for the realised value, the variable cannot be used, and using the realised value anyway is a
leak that looks like skill — the model gets tomorrow's temperature for free and reports a MAE
no production system could reproduce.

For reproducibility in this lab we pass the realised temperatures, so the numbers here are a
mild optimistic bound: they price what temperature is worth *given a perfect forecast of it*.
`data/temp_forecast_28d.csv` holds the actual day-ahead forecast series; substituting it is
the fast-finisher extension, and the MAE gap you measure is the cost of the weather
forecast's own error propagating into yours.

In [ ]:
# TODO:
#   1. forecast_sarimax(sar, H, Xte, alpha=0.10)  -> yhat, lo, hi at 90%
#   2. forecast_ets(ets, H)
#   3. one figure: the last 60 training days, the 28 held-out actuals, both forecasts,
#      and the SARIMAX 90% band shaded.
#
# HINT: get_forecast needs EXACTLY `steps` rows of future exog. If it raises, count the rows
#       in your Xte. And read the paragraph above before you reach for the realised
#       temperatures in production code.

### Task 5 — the scoreboard: accuracy, bias and interval width

In [ ]:
from tayyar.eval.metrics import all_metrics

# TODO:
#   1. fit the SAME SARIMAX with exog=None — this prices what temperature is worth.
#   2. build the seasonal-naive forecast at m=7 (last week repeated four times).
#   3. one table: MAE, RMSE, MAPE_%, MASE and bias for all four forecasts.
#      all_metrics(te, forecast, tr, 7) gives you the row; MASE needs the training series.
#   4. 90% coverage AND mean interval width for every model that produces an interval.
#
# HINT: coverage on its own is not a quality measure. An interval 10,000 MW wide will cover
#       100% of anything and be worth nothing. Report coverage and width together — Module 5
#       makes this precise with the interval score.

### Mini-quiz — answer before you open the solution notebook

1. What do `p`, `d` and `q` control in an ARIMA model?
2. What is `m` for a daily cycle on hourly data?
3. Why must exogenous regressors be known at forecast time?
4. What does Ljung-Box `p < 0.05` on the residuals mean?
5. When is a damped trend preferable to an undamped one?

### Task 6 — commit

The deliverable is the comparison table with the bias and interval-width columns intact, plus
one honest paragraph on the Ljung-Box result. Do not drop the failing diagnostic from the
write-up.

```
git add -A && git commit -m "feat(models): SARIMAX + ETS baselines with intervals and diagnostics"
```